<a href="https://colab.research.google.com/github/mfran3/EPP_EOI_FinalProject_Franke/blob/main/closed_airport_predictions_20260701.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Inference using pretrained YOLOv11 model on NAIP chips at closed airports

!pip install earthengine-api geemap ultralytics -q

import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import json
import os
from ultralytics import YOLO
from google.colab import drive
drive.mount('/content/drive')

ee.Authenticate()
ee.Initialize(project='your_project')
model = YOLO("/content/drive/MyDrive/HRPlanes_runs_test/yolo11m_run_3/weights/best.pt")

# Print the elevation of Mount Everest
# This is just a test to ensure initializing the library worked properly

dem = ee.Image('USGS/SRTMGL1_003')
xy = ee.Geometry.Point([86.9250, 27.9881])
elev = dem.sample(xy, 30).first().get('elevation').getInfo()
print('Mount Everest elevation (m):', elev)

# load .csv

# expected CSV format: name,lon,lat


df = pd.read_csv('/content/drive/MyDrive/closed_lc_final.csv')
print(f"Loaded {len(df)} points")

# Validate required columns
assert 'name' in df.columns, "CSV must have a 'Name' column"
assert 'longitude'  in df.columns, "CSV must have a 'longitude' column"
assert 'latitude'  in df.columns, "CSV must have a 'latitude' column"

# parameters

RADIUS_METERS = 250       # buffer around each point
NAIP_SCALE    = 0.6       # NAIP native resolution (m/px)
CONF          = 0.50      # detection confidence threshold
IOU           = 0.50      # NMS IoU threshold
DATE_START    = "2012-01-01"
DATE_END      = "2024-01-01"
OUTPUT_DIR    = '/content/drive/MyDrive/naip_predictions'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# helper functions

def get_naip_chip(longitude, latitude, radius_m, scale):
    point = ee.Geometry.Point([lon, lat])
    aoi   = point.buffer(radius_m).bounds()

    collection = ee.ImageCollection("USDA/NAIP/DOQQ") \
        .filterBounds(aoi) \
        .filterDate(DATE_START, DATE_END)

    if collection.size().getInfo() == 0:
        raise ValueError(f"No NAIP imagery found at ({lon}, {lat})")

    naip = collection \
        .mosaic() \
        .select(["R", "G", "B"]) \
        .clip(aoi)

    return geemap.ee_to_numpy(naip, region=aoi, scale=scale), aoi

def run_inference(img_array):
    results = model.predict(
        source=img_array,
        conf=CONF,
        iou=IOU,
        save=False,
        verbose=False,
    )
    return results[0]

def save_annotated_image(img_array, result, name, output_dir):
    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(img_array)

    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf = box.conf[0].item()
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor='red', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(
            x1, y1-5,
            f'{conf:.2f}',
            color='white', fontsize=8,
            bbox=dict(facecolor='red', alpha=0.6, pad=1)
        )

    ax.set_title(f"{name} — {len(result.boxes)} aircraft detected")
    ax.axis('off')
    plt.tight_layout()

    save_path = os.path.join(output_dir, f"{name.replace(' ', '_')}.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    return save_path

# prediction loop

records = []

for _, row in df.iterrows():
    name = row['name']
    lon  = row['longitude']
    lat  = row['latitude']

    print(f"\nProcessing: {name} ({lon}, {lat})")

    try:
        # Pull NAIP chip
        img_array, aoi = get_naip_chip(lon, lat, RADIUS_METERS, NAIP_SCALE)
        print(f"  Image shape: {img_array.shape}")

        # Run inference
        result = run_inference(img_array)
        n_detected = len(result.boxes)
        print(f"  Aircraft detected: {n_detected}")

        # Save annotated image
        img_path = save_annotated_image(img_array, result, name, OUTPUT_DIR)

        # Collect per-detection rows
        if n_detected > 0:
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                conf = box.conf[0].item()
                records.append({
                    "name":       name,
                    "longitude":  lon,
                    "latitude":   lat,
                    "confidence": round(conf, 4),
                    "x1": round(x1, 1), "y1": round(y1, 1),
                    "x2": round(x2, 1), "y2": round(y2, 1),
                    "img_path":   img_path,
                    "status":     "success",
                })
        else:
            records.append({
                "name": name, "lon": lon, "lat": lat,
                "confidence": None,
                "x1": None, "y1": None, "x2": None, "y2": None,
                "img_path": img_path,
                "status": "success — no detections",
            })

    except Exception as e:
        print(f"  ERROR: {e}")
        records.append({
            "name": name, "lon": lon, "lat": lat,
            "confidence": None,
            "x1": None, "y1": None, "x2": None, "y2": None,
            "img_path": None,
            "status": f"error: {e}",
        })


# save results as a .csv

results_df = pd.DataFrame(records)
results_csv = os.path.join(OUTPUT_DIR, 'detection_results.csv')
results_df.to_csv(results_csv, index=False)

print(f"\nResults saved to: {results_csv}")
print(results_df.head())


# summary table

summary = results_df.groupby('name').agg(
    detections=('confidence', 'count'),
    avg_confidence=('confidence', 'mean'),
    status=('status', 'first')
).reset_index()

print("\n--- Detection Summary ---")
print(summary.to_string(index=False))